# 08 Stage 1 Component Adapter Probe

역할: SAM/RobustSAM candidate mask를 그대로 graph node로 쓰지 않고, DINO feature와 clean normal prototype memory를 이용해 domain-specific component candidate로 정규화하는 Stage 1 adapter를 검증한다.

현재 Stage 1은 두 단계로 나눈다. 첫째, mask-level descriptor/prototype adapter로 reliable pseudo mask를 고른다. 둘째, reliable mask membership을 prior로 사용해 8-neighbor DINO patch graph에서 masked patch prototype prediction sanity check를 수행한다. 이 노트북은 anomaly detector가 아니라 component boundary adaptation probe다.


## Cell Role: Repo Setup

Colab에서는 `/content/ReGraM` checkout을 사용하고, 로컬에서는 현재 repo root를 찾는다. Stage 1 핵심 로직은 `src/stage1_adapter`에 있고, 노트북은 실행과 시각화만 담당한다.

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import math
import os
import shutil
import subprocess
import sys

import pandas as pd
from IPython.display import display

REPO_URL = 'https://github.com/outSeop/ReGraM.git'


def is_regram_root(path: Path) -> bool:
    return (
        (path / 'experiments' / 'validation' / 'condition_shift_baseline').exists()
        and (path / 'manifests').exists()
    )


colab_checkout = Path('/content/ReGraM')
if Path('/content').exists():
    if (colab_checkout / '.git').exists():
        subprocess.run(['git', '-C', str(colab_checkout), 'fetch', 'origin', 'main'], check=True)
        subprocess.run(
            [
                'git',
                '-C',
                str(colab_checkout),
                'checkout',
                'FETCH_HEAD',
                '--',
                'experiments/validation/condition_shift_baseline/src',
                'experiments/validation/condition_shift_baseline/configs',
                'experiments/validation/condition_shift_baseline/notebook/08_stage1_component_adapter_probe.ipynb',
            ],
            check=True,
        )
    else:
        subprocess.run(['git', 'clone', REPO_URL, str(colab_checkout)], check=True)

cwd = Path.cwd().resolve()
repo_candidates = [colab_checkout, cwd, *cwd.parents]
REPO_ROOT = next((candidate.resolve() for candidate in repo_candidates if candidate.exists() and is_regram_root(candidate)), None)
if REPO_ROOT is None:
    raise FileNotFoundError('ReGraM checkout not found')

EXP_ROOT = REPO_ROOT / 'experiments' / 'validation' / 'condition_shift_baseline'
SRC_ROOT = EXP_ROOT / 'src'
for import_root in [SRC_ROOT, SRC_ROOT / 'orchestration']:
    if str(import_root) not in sys.path:
        sys.path.insert(0, str(import_root))

print('REPO_ROOT =', REPO_ROOT)
print('EXP_ROOT =', EXP_ROOT)


## Cell Role: Settings And Data Readiness

clean normal grounding mask에서 prototype memory를 만든다. query candidate는 06번에서 생성한 RobustSAM shifted masks를 우선 사용한다. query mask가 아직 없으면 clean self-probe만 수행하고, 06을 먼저 실행하라는 안내를 출력한다.

In [ ]:
CATEGORY = 'breakfast_box'
SAMPLE_IDS = ['000.png', '001.png', '002.png']
QUERY_BACKEND = 'robustsam'
QUERY_CONDITIONS = ['brightness_high', 'gaussian_blur_high', 'gaussian_noise_high', 'position_shift_high']
MAX_PROTOTYPES = 8
FEATURE_BACKBONE = 'dinov2_vits14'
IMAGE_SIZE = 224
RING_RADIUS = 2

RAW_LOCO_ROW_ROOT = REPO_ROOT / 'data' / 'row'
RAW_LOCO_ROOT = RAW_LOCO_ROW_ROOT / 'mvtec_loco_anomaly_detection'
GROUNDING_MASK_ROOT = REPO_ROOT / 'external' / 'UniVAD' / 'masks' / 'mvtec_loco_caption'
SHIFTED_PROBE_ROOT = EXP_ROOT / 'reports' / 'shifted_grounding_probe'
GENERATED_MASK_ROOT = SHIFTED_PROBE_ROOT / 'generated_masks' / QUERY_BACKEND
SHIFTED_IMAGE_ROOT = SHIFTED_PROBE_ROOT / 'images'

PROBE_ROOT = EXP_ROOT / 'reports' / 'stage1_component_adapter_probe'
PROTOTYPE_JSON = PROBE_ROOT / f'{CATEGORY}_{QUERY_BACKEND}_stage1_prototypes.json'
DESCRIPTOR_JSON = PROBE_ROOT / f'{CATEGORY}_{QUERY_BACKEND}_stage1_descriptors.json'
SCORE_CSV = PROBE_ROOT / f'{CATEGORY}_{QUERY_BACKEND}_stage1_candidate_scores.csv'
SUMMARY_CSV = PROBE_ROOT / f'{CATEGORY}_{QUERY_BACKEND}_stage1_condition_summary.csv'
MASK_NORMALIZATION_JSON = PROBE_ROOT / f'{CATEGORY}_{QUERY_BACKEND}_stage1_mask_normalization.json'
PROBE_ROOT.mkdir(parents=True, exist_ok=True)

DRIVE_RAW_TAR = Path('/content/drive/MyDrive/ReGraM/data/row/mvtec_loco_anomaly_detection.tar.gz')
DRIVE_MASK_ROOT = Path('/content/drive/MyDrive/ReGraM/univad_masks/mvtec_loco_caption')


def mount_drive_if_available() -> None:
    if not Path('/content').exists() or Path('/content/drive/MyDrive').exists():
        return
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as exc:
        print(f'Google Drive mount skipped: {type(exc).__name__}: {exc}')


def normalize_dataset_root(parent: Path, dataset_name: str) -> Path:
    direct = parent / dataset_name
    nested_candidates = [parent / 'content' / dataset_name, parent / 'data' / 'row' / dataset_name]
    if direct.exists():
        return direct
    for candidate in nested_candidates:
        if candidate.exists():
            direct.parent.mkdir(parents=True, exist_ok=True)
            print(f'normalize dataset root: {candidate} -> {direct}')
            shutil.move(str(candidate), str(direct))
            return direct
    return direct


def restore_raw_loco_from_drive_if_needed() -> None:
    global RAW_LOCO_ROOT
    RAW_LOCO_ROOT = normalize_dataset_root(RAW_LOCO_ROW_ROOT, 'mvtec_loco_anomaly_detection')
    if (RAW_LOCO_ROOT / CATEGORY / 'test' / 'good').exists():
        return
    mount_drive_if_available()
    if not DRIVE_RAW_TAR.exists():
        print(f'raw LOCO Drive tar not found: {DRIVE_RAW_TAR}')
        return
    RAW_LOCO_ROW_ROOT.mkdir(parents=True, exist_ok=True)
    print(f'extract raw LOCO from Drive: {DRIVE_RAW_TAR} -> {RAW_LOCO_ROW_ROOT}')
    subprocess.run(['tar', '-xf', str(DRIVE_RAW_TAR), '-C', str(RAW_LOCO_ROW_ROOT)], check=True)
    RAW_LOCO_ROOT = normalize_dataset_root(RAW_LOCO_ROW_ROOT, 'mvtec_loco_anomaly_detection')


def clean_image_path(image_id: str) -> Path:
    return RAW_LOCO_ROOT / CATEGORY / 'test' / 'good' / image_id


def clean_mask_path(image_path: Path) -> Path:
    rel_path = image_path.relative_to(RAW_LOCO_ROOT / CATEGORY)
    return GROUNDING_MASK_ROOT / CATEGORY / rel_path.with_suffix('') / 'grounding_mask.png'


def restore_clean_masks_from_drive_if_needed(image_paths: list[Path]) -> None:
    missing = [path for path in image_paths if not clean_mask_path(path).exists()]
    if not missing:
        return
    mount_drive_if_available()
    copied = 0
    category_drive_root = DRIVE_MASK_ROOT / CATEGORY
    category_session_root = GROUNDING_MASK_ROOT / CATEGORY
    if category_drive_root.exists() and not category_session_root.exists():
        category_session_root.parent.mkdir(parents=True, exist_ok=True)
        print(f'restore clean masks category from Drive: {category_drive_root} -> {category_session_root}')
        shutil.copytree(category_drive_root, category_session_root, dirs_exist_ok=True)
    for image_path in missing:
        session_mask = clean_mask_path(image_path)
        rel_path = session_mask.relative_to(category_session_root)
        drive_candidates = [
            category_drive_root / rel_path,
            category_drive_root / 'test' / 'good' / image_path.stem / 'grounding_mask.png',
            category_drive_root / image_path.stem / 'grounding_mask.png',
        ]
        drive_mask = next((candidate for candidate in drive_candidates if candidate.exists()), None)
        if drive_mask is None:
            continue
        session_mask.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(drive_mask, session_mask)
        copied += 1
    remaining = sum(not clean_mask_path(path).exists() for path in missing)
    print(f'restored clean masks from Drive: copied={copied} remaining={remaining}')


def clean_row_diagnostics() -> pd.DataFrame:
    rows = []
    for image_id in SAMPLE_IDS:
        image_path = clean_image_path(image_id)
        mask_path = clean_mask_path(image_path)
        rows.append(
            {
                'image_id': image_id,
                'image_exists': image_path.exists(),
                'mask_exists': mask_path.exists(),
                'image_path': str(image_path),
                'mask_path': str(mask_path),
                'drive_mask_root_exists': (DRIVE_MASK_ROOT / CATEGORY).exists(),
                'drive_raw_tar_exists': DRIVE_RAW_TAR.exists(),
            }
        )
    return pd.DataFrame(rows)


def shifted_image_path(condition: str, image_id: str) -> Path:
    return SHIFTED_IMAGE_ROOT / CATEGORY / condition / image_id


def generated_query_mask_path(condition: str, image_id: str) -> Path | None:
    output_dir = GENERATED_MASK_ROOT / CATEGORY / condition
    if not output_dir.exists():
        return None
    candidates = sorted(output_dir.rglob(f'{Path(image_id).stem}/grounding_mask.png'))
    return candidates[0] if candidates else None


restore_raw_loco_from_drive_if_needed()
source_paths = [clean_image_path(image_id) for image_id in SAMPLE_IDS]
restore_clean_masks_from_drive_if_needed(source_paths)
settings_df = pd.DataFrame([
    {
        'category': CATEGORY,
        'sample_ids': ', '.join(SAMPLE_IDS),
        'query_backend': QUERY_BACKEND,
        'query_conditions': ', '.join(QUERY_CONDITIONS),
        'feature_backbone': FEATURE_BACKBONE,
        'prototype_json': str(PROTOTYPE_JSON),
        'score_csv': str(SCORE_CSV),
    }
])
display(settings_df)


## Cell Role: Build Candidate Source Table

clean rows는 prototype memory와 self sanity check에 사용한다. shifted rows는 06번이 생성한 RobustSAM mask가 있는 경우에만 Stage 1 scoring에 들어간다.

In [ ]:
candidate_rows = []
missing_clean = []
for image_id in SAMPLE_IDS:
    image_path = clean_image_path(image_id)
    mask_path = clean_mask_path(image_path)
    if image_path.exists() and mask_path.exists():
        candidate_rows.append(
            {
                'base_image_id': image_id,
                'condition': 'clean_reference',
                'split': 'clean',
                'image_path': str(image_path),
                'mask_path': str(mask_path),
                'candidate_source': 'clean_grounding_mask',
            }
        )
    else:
        missing_clean.append(
            {
                'image_id': image_id,
                'image_exists': image_path.exists(),
                'mask_exists': mask_path.exists(),
                'image_path': str(image_path),
                'mask_path': str(mask_path),
            }
        )

if missing_clean:
    print(f'missing clean reference rows: {len(missing_clean)}/{len(SAMPLE_IDS)}')
    display(pd.DataFrame(missing_clean))

missing_query = []
for condition in QUERY_CONDITIONS:
    for image_id in SAMPLE_IDS:
        image_path = shifted_image_path(condition, image_id)
        mask_path = generated_query_mask_path(condition, image_id)
        if image_path.exists() and mask_path is not None and mask_path.exists():
            candidate_rows.append(
                {
                    'base_image_id': image_id,
                    'condition': condition,
                    'split': 'shifted_query',
                    'image_path': str(image_path),
                    'mask_path': str(mask_path),
                    'candidate_source': f'{QUERY_BACKEND}_generated_mask',
                }
            )
        else:
            missing_query.append((condition, image_id))

if missing_query:
    print(f'missing generated query masks: {len(missing_query)}. Run 06 first for full shifted probe. examples={missing_query[:5]}')

candidate_df = pd.DataFrame(candidate_rows)
if candidate_df.empty:
    diagnostics = clean_row_diagnostics()
    display(diagnostics)
    raise FileNotFoundError(
        'No clean/query candidate rows found. At least one clean reference image and grounding mask is required. '
        'Run 01 setup or make sure Drive contains raw LOCO tar and clean masks. '
        f'raw_root={RAW_LOCO_ROOT} mask_root={GROUNDING_MASK_ROOT / CATEGORY} '
        f'drive_raw_tar={DRIVE_RAW_TAR} drive_mask_root={DRIVE_MASK_ROOT / CATEGORY}'
    )
display(candidate_df)


## Cell Role: DINO Feature Extractor

DINOv2 patch tokens를 mask 내부/ring descriptor에 사용한다. 첫 실행에서는 torch hub download가 필요할 수 있다.

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device =', device)

dino_model = torch.hub.load('facebookresearch/dinov2', FEATURE_BACKBONE).to(device).eval()
dino_mean = torch.tensor([0.485, 0.456, 0.406], dtype=torch.float32, device=device).view(1, 3, 1, 1)
dino_std = torch.tensor([0.229, 0.224, 0.225], dtype=torch.float32, device=device).view(1, 3, 1, 1)


def preprocess_dino(image: Image.Image) -> torch.Tensor:
    image = image.convert('RGB').resize((IMAGE_SIZE, IMAGE_SIZE), resample=Image.Resampling.BICUBIC)
    array = np.asarray(image, dtype=np.float32) / 255.0
    tensor = torch.from_numpy(array).permute(2, 0, 1).unsqueeze(0).to(device)
    return (tensor - dino_mean) / dino_std


@torch.no_grad()
def extract_dino_feature_map(image_path: Path) -> np.ndarray:
    image = Image.open(image_path).convert('RGB')
    output = dino_model.forward_features(preprocess_dino(image))
    tokens = output['x_norm_patchtokens'] if isinstance(output, dict) else output
    tokens = tokens.squeeze(0).detach().cpu().float()
    grid = int(math.sqrt(tokens.shape[0]))
    if grid * grid != tokens.shape[0]:
        raise ValueError(f'DINO token count is not square: {tokens.shape[0]}')
    return tokens.reshape(grid, grid, tokens.shape[-1]).numpy()

print('DINO backbone ready:', FEATURE_BACKBONE)


## Cell Role: Build Clean Normal Component Prototypes

clean normal candidate masks에서 descriptor를 만들고 deterministic prototype memory를 구성한다. prototype은 semantic class가 아니라 domain-specific role memory다.

In [ ]:
from graph_probe.component_io import save_json
from relation.grounding_mask_cluster import raw_masks_from_label_image
from stage1_adapter import (
    CandidateMaskNormalizationConfig,
    build_component_prototypes,
    describe_candidate_masks,
    normalize_candidate_masks,
    score_candidate_against_prototypes,
    summarize_adapter_scores,
)

MASK_NORMALIZATION_CONFIG = CandidateMaskNormalizationConfig(
    max_mask_area_ratio=0.30,
    min_mask_area_ratio=0.001,
    small_cluster_area_ratio=0.006,
    min_cluster_members=3,
    min_cluster_union_area_ratio=0.004,
    max_cluster_union_area_ratio=0.25,
    max_centroid_dist_ratio=0.10,
    max_bbox_gap_ratio=0.04,
)

feature_cache: dict[str, np.ndarray] = {}
raw_mask_cache: dict[tuple[str, tuple[int, int], str], list[dict]] = {}
mask_normalization_summaries: dict[str, dict] = {}


def feature_map_for(path: Path) -> np.ndarray:
    key = str(path)
    if key not in feature_cache:
        feature_cache[key] = extract_dino_feature_map(path)
    return feature_cache[key]


def raw_masks_for(mask_path: Path, *, image_shape: tuple[int, int], normalize: bool = True) -> list[dict]:
    mode = 'normalized' if normalize else 'raw'
    key = (str(mask_path), image_shape, mode)
    if key not in raw_mask_cache:
        raw_masks = raw_masks_from_label_image(Image.open(mask_path).convert('RGB'))
        if normalize:
            normalized_masks, summary = normalize_candidate_masks(
                raw_masks,
                image_shape=image_shape,
                config=MASK_NORMALIZATION_CONFIG,
            )
            raw_mask_cache[key] = normalized_masks
            mask_normalization_summaries[str(mask_path)] = summary
        else:
            for raw_mask in raw_masks:
                raw_mask.setdefault('use_for_patch_graph', True)
                raw_mask.setdefault('use_as_component_node', False)
                raw_mask.setdefault('normalization_type', 'raw_candidate')
            raw_mask_cache[key] = raw_masks
    return raw_mask_cache[key]


def mask_metadata_for(mask_path: Path, *, image_shape: tuple[int, int]) -> dict[str, dict]:
    return {str(mask.get('mask_id')): mask for mask in raw_masks_for(mask_path, image_shape=image_shape)}


def describe_row(row: pd.Series) -> list:
    image_path = Path(row['image_path'])
    mask_path = Path(row['mask_path'])
    image = Image.open(image_path).convert('RGB')
    return describe_candidate_masks(
        image_id=str(row['base_image_id']),
        feature_map=feature_map_for(image_path),
        raw_masks=raw_masks_for(mask_path, image_shape=(image.height, image.width)),
        image_shape=(image.height, image.width),
        source=str(row['candidate_source']),
        ring_radius=RING_RADIUS,
    )

clean_descriptors = []
for _, row in candidate_df[candidate_df['split'].eq('clean')].iterrows():
    clean_descriptors.extend(describe_row(row))

prototypes = build_component_prototypes(clean_descriptors, max_prototypes=MAX_PROTOTYPES)
prototype_doc = {
    'probe': 'stage1_component_adapter',
    'category': CATEGORY,
    'feature_backbone': FEATURE_BACKBONE,
    'num_clean_descriptors': len(clean_descriptors),
    'num_prototypes': len(prototypes),
    'prototypes': [prototype.to_dict() for prototype in prototypes],
}
save_json(prototype_doc, PROTOTYPE_JSON)
print(f'saved prototypes: {PROTOTYPE_JSON}')
display(pd.DataFrame([prototype.to_dict() for prototype in prototypes]).drop(columns=['mean_vector', 'inside_mean', 'ring_mean', 'debug'], errors='ignore'))


## Cell Role: Score Clean And Query Candidate Masks

각 candidate mask를 clean prototype memory에 매칭한다. 출력은 prototype compatibility, reliability, invalid proposal type이다. `clean_reference`는 sanity check, shifted condition은 Stage 1의 실제 관심 대상이다.

In [ ]:
descriptor_docs = []
score_rows = []
for _, row in candidate_df.iterrows():
    image_for_meta = Image.open(Path(row['image_path'])).convert('RGB')
    mask_meta = mask_metadata_for(Path(row['mask_path']), image_shape=(image_for_meta.height, image_for_meta.width))
    descriptors = describe_row(row)
    row_scores = []
    for descriptor in descriptors:
        descriptor_doc = descriptor.to_dict()
        descriptor_doc.update({'condition': row['condition'], 'split': row['split']})
        descriptor_docs.append(descriptor_doc)
        score = score_candidate_against_prototypes(descriptor, prototypes)
        metadata = mask_meta.get(str(descriptor.candidate_id), {})
        score.update(
            {
                'normalization_type': metadata.get('normalization_type', 'unknown'),
                'use_for_patch_graph': bool(metadata.get('use_for_patch_graph', True)),
                'use_as_component_node': bool(metadata.get('use_as_component_node', False)),
                'condition': row['condition'],
                'split': row['split'],
                'base_image_id': row['base_image_id'],
                'image_path': row['image_path'],
                'mask_path': row['mask_path'],
                'candidate_source': row['candidate_source'],
            }
        )
        score_rows.append(score)
        row_scores.append(score)
    print(row['condition'], row['base_image_id'], summarize_adapter_scores(row_scores))

score_df = pd.DataFrame(score_rows)
score_df.to_csv(SCORE_CSV, index=False)
save_json({'descriptors': descriptor_docs, 'mask_normalization': mask_normalization_summaries}, DESCRIPTOR_JSON)
save_json({'config': MASK_NORMALIZATION_CONFIG.to_dict(), 'rows': mask_normalization_summaries}, MASK_NORMALIZATION_JSON)
summary_df = score_df.groupby('condition').agg(
    num_candidates=('candidate_id', 'count'),
    mean_reliability=('reliability', 'mean'),
    median_reliability=('reliability', 'median'),
    valid_components=('node_type', lambda values: sum(str(value) == 'valid_component' for value in values)),
    invalid_supermasks=('node_type', lambda values: sum(str(value) == 'invalid_supermask' for value in values)),
    invalid_fragments=('node_type', lambda values: sum(str(value) == 'invalid_fragment' for value in values)),
    patch_graph_candidates=('use_for_patch_graph', 'sum'),
    component_node_candidates=('use_as_component_node', 'sum'),
).reset_index()
summary_df.to_csv(SUMMARY_CSV, index=False)
normalization_df = pd.DataFrame([{'mask_path': path, **summary} for path, summary in mask_normalization_summaries.items()])
print(f'saved scores: {SCORE_CSV}')
print(f'saved mask normalization: {MASK_NORMALIZATION_JSON}')
if not normalization_df.empty:
    display(normalization_df[['num_input_masks', 'num_output_masks', 'num_thing_candidates', 'num_stuff_clusters', 'num_excluded_large', 'num_excluded_small']].describe())
display(score_df[['condition', 'base_image_id', 'candidate_id', 'normalization_type', 'use_for_patch_graph', 'use_as_component_node', 'node_type', 'best_prototype_id', 'reliability', 'max_prototype_similarity', 'prototype_margin', 'prototype_entropy_norm', 'area_ratio', 'contained_neighbor_count']])
display(summary_df)


## Cell Role: Mask-Aware Patch Graph Probe

reliable candidate mask를 SAM pseudo mask prior로 사용한다. 각 DINO patch의 teacher prototype assignment를 만들고, target patch를 가렸다고 가정한 뒤 주변 8-neighbor patch와 same-mask/cross-boundary relation으로 target assignment를 예측한다. 아직 학습 모델은 넣지 않고, learnable adapter로 가기 전 신호가 있는지 보는 deterministic baseline이다.


In [ ]:
from stage1_adapter import PatchGraphConfig, patch_region_masks, run_masked_patch_prototype_probe

PATCH_GRAPH_JSON = PROBE_ROOT / f'{CATEGORY}_{QUERY_BACKEND}_stage1_patch_graph_probe.json'
PATCH_GRAPH_SUMMARY_CSV = PROBE_ROOT / f'{CATEGORY}_{QUERY_BACKEND}_stage1_patch_graph_summary.csv'
TEACHER_LAMBDAS = [0.0, 0.1, 0.2, 0.3, 0.5]
SMOOTHING_MODES = ['none', 'global_same_mask', 'local_same_mask', 'boundary_weighted_local']


def make_patch_config(
    *,
    mask_source: str,
    target_type: str,
    smoothing_mode: str,
    smoothing_lambda: float,
) -> PatchGraphConfig:
    if mask_source == 'no_mask':
        return PatchGraphConfig(
            reliable_selection_mode='none',
            target_type=target_type,
            teacher_smoothing_mode=smoothing_mode,
            teacher_smoothing_lambda=smoothing_lambda,
            teacher_inner_smoothing_lambda=0.1,
            teacher_boundary_smoothing_lambda=smoothing_lambda,
            prototype_temperature=0.07,
        )
    if mask_source == 'component_node_only':
        mask_usage_field = 'use_as_component_node'
    elif mask_source == 'normalized_topk_mask':
        mask_usage_field = 'use_for_patch_graph'
    else:
        raise ValueError(f'Unsupported mask_source={mask_source}')
    return PatchGraphConfig(
        reliable_selection_mode='top_k',
        top_k_reliable_masks=8,
        min_reliability=0.35,
        mask_usage_field=mask_usage_field,
        target_type=target_type,
        teacher_smoothing_mode=smoothing_mode,
        teacher_smoothing_lambda=smoothing_lambda,
        teacher_inner_smoothing_lambda=0.1,
        teacher_boundary_smoothing_lambda=smoothing_lambda,
        valid_node_types=('valid_component', 'small_detail', 'low_reliability_candidate'),
        hard_excluded_node_types=('invalid_supermask', 'invalid_fragment'),
        max_area_ratio=0.30,
        min_area_ratio=0.001,
        same_mask_bonus=1.0,
        cross_boundary_weight=0.25,
        unlabeled_neighbor_weight=0.5,
        prototype_temperature=0.07,
    )


PATCH_GRAPH_RUNS = []
PATCH_GRAPH_RUNS.append(('dino_only', 'no_mask', 'none', 0.0, make_patch_config(mask_source='no_mask', target_type='dino_only', smoothing_mode='none', smoothing_lambda=0.0)))
for mask_source in ['component_node_only', 'normalized_topk_mask']:
    PATCH_GRAPH_RUNS.append(('dino_only', mask_source, 'none', 0.0, make_patch_config(mask_source=mask_source, target_type='dino_only', smoothing_mode='none', smoothing_lambda=0.0)))
    for smoothing_mode in SMOOTHING_MODES:
        lambdas = [0.0] if smoothing_mode == 'none' else TEACHER_LAMBDAS
        for smoothing_lambda in lambdas:
            PATCH_GRAPH_RUNS.append(
                (
                    'mask_smoothed',
                    mask_source,
                    smoothing_mode,
                    smoothing_lambda,
                    make_patch_config(
                        mask_source=mask_source,
                        target_type='mask_smoothed',
                        smoothing_mode=smoothing_mode,
                        smoothing_lambda=smoothing_lambda,
                    ),
                )
            )

patch_probe_results = {}
patch_summary_rows = []
for target_type, mask_source, smoothing_mode, smoothing_lambda, patch_config in PATCH_GRAPH_RUNS:
    for _, row in candidate_df.iterrows():
        key = (target_type, mask_source, smoothing_mode, float(smoothing_lambda), str(row['condition']), str(row['base_image_id']))
        row_scores = score_df[(score_df['condition'] == row['condition']) & (score_df['base_image_id'] == row['base_image_id'])].to_dict('records')
        image_path = Path(row['image_path'])
        mask_path = Path(row['mask_path'])
        image = Image.open(image_path).convert('RGB')
        result = run_masked_patch_prototype_probe(
            feature_map=feature_map_for(image_path),
            raw_masks=raw_masks_for(mask_path, image_shape=(image.height, image.width)),
            candidate_scores=row_scores,
            prototypes=prototypes,
            image_shape=(image.height, image.width),
            config=patch_config,
        )
        patch_probe_results[key] = result
        teacher_quality = result.summary['teacher_quality']
        dino_teacher_quality = result.summary['dino_teacher_quality']
        patch_summary_rows.append(
            {
                'target_type': target_type,
                'mask_source': mask_source,
                'smoothing_mode': smoothing_mode,
                'lambda': float(smoothing_lambda),
                'condition': key[4],
                'base_image_id': key[5],
                **{name: value for name, value in result.summary.items() if name not in {'edge_summary', 'edge_metrics', 'teacher_quality', 'dino_teacher_quality', 'membership_debug', 'config'}},
                'teacher_same_mask_consistency': teacher_quality['same_mask_consistency'],
                'teacher_cross_boundary_contrast': teacher_quality['cross_boundary_contrast'],
                'teacher_mask_agreement': teacher_quality['mask_teacher_agreement'],
                'teacher_cross_mask_separation': teacher_quality['cross_mask_teacher_separation'],
                'teacher_boundary_sharpness': teacher_quality['boundary_sharpness'],
                'dino_teacher_same_mask_consistency': dino_teacher_quality['same_mask_consistency'],
                'dino_teacher_cross_boundary_contrast': dino_teacher_quality['cross_boundary_contrast'],
                'dino_teacher_boundary_sharpness': dino_teacher_quality['boundary_sharpness'],
                'num_same_mask_edges': result.summary['edge_summary']['num_same_mask_edges'],
                'num_cross_boundary_edges': result.summary['edge_summary']['num_cross_boundary_edges'],
                'num_unlabeled_edges': result.summary['edge_summary']['num_unlabeled_edges'],
                'num_reliable_masks': result.summary['membership_debug']['num_reliable_masks'],
                'selected_candidate_ids': ', '.join(result.summary['membership_debug']['selection']['selected_candidate_ids']),
            }
        )

patch_summary_df = pd.DataFrame(patch_summary_rows)
patch_summary_df.to_csv(PATCH_GRAPH_SUMMARY_CSV, index=False)
save_json(
    {
        'probe': 'stage1_mask_smoothed_teacher_patch_graph',
        'category': CATEGORY,
        'query_backend': QUERY_BACKEND,
        'runs': [
            {
                'target_type': target_type,
                'mask_source': mask_source,
                'smoothing_mode': smoothing_mode,
                'lambda': float(smoothing_lambda),
                'config': config.to_dict(),
            }
            for target_type, mask_source, smoothing_mode, smoothing_lambda, config in PATCH_GRAPH_RUNS
        ],
        'rows': [
            {
                'target_type': target_type,
                'mask_source': mask_source,
                'smoothing_mode': smoothing_mode,
                'lambda': float(smoothing_lambda),
                'condition': condition,
                'base_image_id': base_image_id,
                'summary': result.summary,
            }
            for (target_type, mask_source, smoothing_mode, smoothing_lambda, condition, base_image_id), result in patch_probe_results.items()
        ],
    },
    PATCH_GRAPH_JSON,
)
print(f'saved patch graph probe: {PATCH_GRAPH_JSON}')
display(patch_summary_df)
display(
    patch_summary_df.groupby(['target_type', 'mask_source', 'smoothing_mode', 'lambda', 'condition'])[
        [
            'teacher_entropy_mean',
            'teacher_mask_agreement',
            'teacher_cross_mask_separation',
            'teacher_boundary_sharpness',
            'teacher_argmax_change_rate',
            'boundary_delta_mean',
            'neighbor_prediction_kl_mean',
            'neighbor_argmax_match_ratio',
            'boundary_patch_match_ratio',
            'same_mask_edge_consistency',
            'cross_boundary_contrast',
        ]
    ].mean().reset_index()
)


## Cell Role: Visualize Mask-Aware Patch Prototype Maps

같은 이미지에서 비교군을 나란히 본다. `SAM only`는 normalized reliable mask membership, `DINO only`는 mask smoothing 없는 prototype assignment, `SAM + DINO`는 reliable mask 내부에서 약하게 smoothed된 teacher, 마지막은 mask-aware 8-neighbor prediction이다.


In [ ]:
import matplotlib.pyplot as plt

In [ ]:
COMPARISON_MASK_SOURCE = 'component_node_only'
COMPARISON_SMOOTHING_MODE = 'boundary_weighted_local'
COMPARISON_LAMBDA = 0.3


def _patch_result(row: pd.Series, *, target_type: str, mask_source: str, smoothing_mode: str, smoothing_lambda: float):
    key = (target_type, mask_source, smoothing_mode, float(smoothing_lambda), str(row['condition']), str(row['base_image_id']))
    return patch_probe_results.get(key)


def draw_stage1_comparison(row: pd.Series) -> None:
    dino_result = _patch_result(row, target_type='dino_only', mask_source='no_mask', smoothing_mode='none', smoothing_lambda=0.0)
    sam_dino_result = _patch_result(
        row,
        target_type='mask_smoothed',
        mask_source=COMPARISON_MASK_SOURCE,
        smoothing_mode=COMPARISON_SMOOTHING_MODE,
        smoothing_lambda=COMPARISON_LAMBDA,
    )
    if dino_result is None or sam_dino_result is None:
        return

    image = Image.open(Path(row['image_path'])).convert('RGB')
    fig, axes = plt.subplots(1, 7, figsize=(25, 4))
    axes[0].imshow(image)
    axes[0].set_title(f"image {row['condition']} / {row['base_image_id']}")

    membership_display = np.ma.masked_where(sam_dino_result.membership < 0, sam_dino_result.membership)
    axes[1].imshow(membership_display, cmap='tab20', interpolation='nearest')
    axes[1].set_title('SAM only\nreliable mask membership')

    axes[2].imshow(dino_result.dino_teacher_assignment, cmap='tab20', interpolation='nearest')
    axes[2].set_title('DINO only\nprototype assignment')

    axes[3].imshow(sam_dino_result.teacher_assignment, cmap='tab20', interpolation='nearest')
    axes[3].set_title(f'SAM + DINO\n{COMPARISON_SMOOTHING_MODE} λ={COMPARISON_LAMBDA}')

    axes[4].imshow(sam_dino_result.teacher_delta_map, cmap='magma', interpolation='nearest')
    axes[4].set_title('SAM injection\nteacher delta')

    boundary_mask = patch_region_masks(sam_dino_result.membership)['boundary']
    boundary_delta = np.ma.masked_where(~boundary_mask, sam_dino_result.teacher_delta_map)
    axes[5].imshow(boundary_delta, cmap='magma', interpolation='nearest')
    axes[5].set_title('SAM injection\nboundary delta')

    axes[6].imshow(sam_dino_result.neighbor_assignment, cmap='tab20', interpolation='nearest')
    axes[6].set_title('SAM + DINO\n8-neighbor prediction')

    for ax in axes:
        ax.axis('off')
    plt.tight_layout()
    plt.show()


PATCH_VIS_ROWS = min(8, len(candidate_df))
for _, row in candidate_df.head(PATCH_VIS_ROWS).iterrows():
    draw_stage1_comparison(row)


## Cell Role: Visualize Stage 1 Reliability

candidate mask bbox를 reliability/node type 색상으로 overlay한다. 이 뷰에서 super-mask, fragment, low reliability candidate가 Stage 2로 넘어가지 않도록 걸러지는지 확인한다.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

NODE_COLORS = {
    'valid_component': '#2ca02c',
    'small_detail': '#1f77b4',
    'low_reliability_candidate': '#ff7f0e',
    'invalid_supermask': '#d62728',
    'invalid_fragment': '#9467bd',
}

if score_df.empty:
    print('No Stage 1 scores to visualize')
else:
    ax = score_df.groupby(['condition', 'node_type']).size().unstack(fill_value=0).plot(kind='bar', stacked=True, figsize=(12, 4))
    ax.set_title('Stage 1 node type distribution')
    ax.set_ylabel('candidate count')
    ax.tick_params(axis='x', rotation=25)
    plt.tight_layout()
    plt.show()

    ax = score_df.boxplot(column='reliability', by='condition', figsize=(10, 4), rot=25)
    ax.set_title('Stage 1 reliability by condition')
    ax.set_ylabel('reliability')
    plt.suptitle('')
    plt.tight_layout()
    plt.show()


def draw_stage1_scores(row: pd.Series, row_scores: pd.DataFrame) -> None:
    image = Image.open(Path(row['image_path'])).convert('RGB')
    fig, ax = plt.subplots(1, 1, figsize=(6, 5))
    ax.imshow(image)
    for _, score in row_scores.iterrows():
        x1, y1, x2, y2 = [float(value) for value in score['bbox']]
        node_type = str(score['node_type'])
        color = NODE_COLORS.get(node_type, '#666666')
        ax.add_patch(patches.Rectangle((x1, y1), max(1, x2 - x1), max(1, y2 - y1), fill=False, edgecolor=color, linewidth=1.5))
        ax.text(x1, y1, f"{node_type} r={score['reliability']:.2f}", color=color, fontsize=7)
    ax.set_title(f"{row['condition']} / {row['base_image_id']}")
    ax.axis('off')
    plt.tight_layout()
    plt.show()

VIS_ROWS = min(8, len(candidate_df))
for _, row in candidate_df.head(VIS_ROWS).iterrows():
    row_scores = score_df[(score_df['condition'] == row['condition']) & (score_df['base_image_id'] == row['base_image_id'])]
    draw_stage1_scores(row, row_scores)


## Cell Role: Interpretation Checklist

확인할 것:

- clean_reference에서 prototype matching과 reliability가 안정적인가?
- shifted RobustSAM candidate 중 큰 tray/blob는 `invalid_supermask` 또는 낮은 reliability로 내려가는가?
- 작은 fragment는 `invalid_fragment` 또는 `small_detail`로 분리되는가?
- valid_component만 Stage 2 relation graph로 넘겼을 때 node 구성이 더 안정적인가?

여기서 신호가 약하면 learnable graph adapter로 가기 전에 descriptor/prototype 정의부터 다시 봐야 한다.